# Cold Diffusion: Inverting Arbitrary Image Transforms Without Noise — Implementation / 구현

**Paper**: Bansal, A., Borgnia, E., Chu, H.-M., Li, J. S., Kazemi, H., Huang, F., Goldblum, M., Geiping, J., & Goldstein, T. (2023). *Proc. NeurIPS*. [arXiv:2208.09392]

## Overview / 개요

이 노트북은 Cold Diffusion의 핵심을 검증한다:
1. **Deterministic forward chain** $D(x_0, t)$: noise 없이 progressive Gaussian blur로 영상을 점진적으로 corrupt.
2. **Train a tiny CNN restorer** $R_\theta(x_t, t) \approx x_0$: $\ell_1$ loss로 학습.
3. **Algorithm 1 vs Algorithm 2 sampling**: 같은 $R$로 두 sampler의 결과를 비교 — Algorithm 2의 stability 우월성 확인.

This notebook verifies Cold Diffusion on a tiny synthetic image dataset:
1. **Deterministic blur forward chain** without any Gaussian noise.
2. **Tiny CNN restorer** trained with $\ell_1$ loss in <1 minute on CPU.
3. **Compare Algorithm 1 (naive) vs Algorithm 2 (improved)** sampling — empirically verify Algorithm 2 is more stable for cold diffusion.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['font.size'] = 10
torch.manual_seed(0)
np.random.seed(0)
rng = np.random.default_rng(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---

## Part 1: Deterministic forward chain D(x, t) / 결정적 forward chain

$D(x, t) = \bar G_t * x$ where $\bar G_t$ = $t$-step iterated Gaussian blur kernel. Equivalent to a single Gaussian blur with bandwidth $\sigma_b\sqrt{t}$.

**핵심**: 이 forward는 *완전히 결정적* — Gaussian noise 없음. 표준 DDPM과 정반대.

`apply_blur(x, t)` is callable as $D$ in Algorithm 1/2 below.

In [ ]:
def apply_blur(x: NDArray[np.float64], t: float, sigma_per_step: float = 0.5) -> NDArray[np.float64]:
    """Deterministic forward operator: apply Gaussian blur with bandwidth proportional to sqrt(t).

    Args:
        x: image (..., H, W).
        t: degradation severity (>= 0; 0 = identity).
        sigma_per_step: blur kernel std per unit t.

    Returns:
        Blurred image, same shape as x.
    """
    if t <= 0:
        return x.copy()
    sigma = sigma_per_step * np.sqrt(max(t, 0))
    return gaussian_filter(x, sigma=(sigma, sigma) if x.ndim == 2 else (0,) + (sigma, sigma), mode='reflect')


def apply_blur_torch(x: torch.Tensor, t: float, sigma_per_step: float = 0.5, device='cpu') -> torch.Tensor:
    """Same forward operator, torch version (for use during training/sampling)."""
    if t <= 0:
        return x.clone()
    # Apply via numpy under the hood (small images, fine on CPU)
    arr = x.detach().cpu().numpy()
    sigma = sigma_per_step * np.sqrt(max(t, 0))
    if arr.ndim == 4:  # (B, C, H, W)
        out = np.empty_like(arr)
        for b in range(arr.shape[0]):
            for c in range(arr.shape[1]):
                out[b, c] = gaussian_filter(arr[b, c], sigma=sigma, mode='reflect')
    elif arr.ndim == 3:  # (C, H, W)
        out = np.empty_like(arr)
        for c in range(arr.shape[0]):
            out[c] = gaussian_filter(arr[c], sigma=sigma, mode='reflect')
    else:
        out = gaussian_filter(arr, sigma=sigma, mode='reflect')
    return torch.tensor(out, dtype=x.dtype, device=device)


# Demonstrate the forward chain on a sample image
from skimage import data
from skimage.transform import resize
from skimage import img_as_float

test_img = resize(img_as_float(data.camera()), (32, 32), preserve_range=True).astype(np.float32)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, t_val in enumerate([0, 1, 2, 4, 8, 16]):
    blurred = apply_blur(test_img, t_val, sigma_per_step=0.5)
    axes[i].imshow(blurred, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f't={t_val}'); axes[i].axis('off')
plt.suptitle(r'Deterministic forward chain $D(x, t) = \bar G_t * x$ (no noise)')
plt.tight_layout(); plt.show()
print('No randomness in this forward chain — same image every run.')

---

## Part 2: Generate tiny training dataset / 작은 학습 데이터셋 생성

200 random synthetic 16×16 images (Gaussian blobs of random position/size) for fast CPU training.

In [ ]:
def make_random_blob_image(size: int, n_blobs: int = 3, seed: int = 0) -> NDArray[np.float32]:
    """Generate a synthetic image with random Gaussian blobs."""
    g = np.random.default_rng(seed)
    yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij')
    img = np.zeros((size, size))
    for _ in range(n_blobs):
        cx, cy = g.uniform(-0.7, 0.7, 2)
        amp = g.uniform(0.3, 0.8)
        width = g.uniform(0.1, 0.3)
        img += amp * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / width ** 2)
    return np.clip(img, 0.0, 1.0).astype(np.float32)


SIZE = 16
N_TRAIN = 200
T_MAX = 8  # max severity in training
training_images = np.stack([make_random_blob_image(SIZE, seed=i) for i in range(N_TRAIN)], axis=0)
print(f'Training set: {training_images.shape}, dtype={training_images.dtype}')

# Visualise a few
fig, axes = plt.subplots(1, 6, figsize=(13, 3))
for i in range(6):
    axes[i].imshow(training_images[i], cmap='gray', vmin=0, vmax=1); axes[i].axis('off')
plt.suptitle('Sample training images')
plt.tight_layout(); plt.show()

---

## Part 3: Tiny restoration CNN $R_\theta(x_t, t) \approx x_0$ / 작은 복원 CNN

Simple 3-layer CNN with $t$ embedded as additional channel. Trained with $\ell_1$ loss as in Eq. 1.

$R$은 단순 3-layer CNN. severity $t$를 broadcast channel로 입력. Eq. 1대로 $\ell_1$ loss로 학습.

In [ ]:
class TinyRestorer(nn.Module):
    """3-layer CNN with severity t injected as additional channel."""

    def __init__(self, n_channels: int = 16) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(2, n_channels, kernel_size=3, padding=1)  # 2 input: x_t + t-channel
        self.conv2 = nn.Conv2d(n_channels, n_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(n_channels, 1, kernel_size=3, padding=1)

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # x_t: (B, 1, H, W), t: (B,) — broadcast t into a channel
        t_channel = t.view(-1, 1, 1, 1).expand(-1, 1, x_t.shape[-2], x_t.shape[-1]) / float(T_MAX)
        h = torch.cat([x_t, t_channel], dim=1)
        h = F.relu(self.conv1(h))
        h = F.relu(self.conv2(h))
        return self.conv3(h)  # predict x_0 directly (not residual)


model = TinyRestorer(n_channels=16).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'TinyRestorer parameter count: {n_params}')

In [ ]:
# Training loop
x_train = torch.tensor(training_images[:, None, :, :], dtype=torch.float32, device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
n_epochs = 200
batch_size = 32
loss_history = []
g_torch = torch.Generator(device=device).manual_seed(0)

for epoch in range(n_epochs):
    # Random t per training sample
    t_batch = torch.rand(N_TRAIN, generator=g_torch, device=device) * T_MAX
    # Apply forward operator x_t = D(x_0, t)
    x_t = torch.empty_like(x_train)
    for i in range(N_TRAIN):
        x_t[i] = apply_blur_torch(x_train[i], float(t_batch[i].item()), sigma_per_step=0.5, device=device)
    perm = torch.randperm(N_TRAIN)
    epoch_loss = 0.0
    for k in range(0, N_TRAIN, batch_size):
        idx = perm[k : k + batch_size]
        pred = model(x_t[idx], t_batch[idx])
        loss = F.l1_loss(pred, x_train[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * idx.shape[0]
    loss_history.append(epoch_loss / N_TRAIN)
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch + 1}: L1 loss = {loss_history[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(loss_history)
ax.set_xlabel('epoch'); ax.set_ylabel('L1 loss')
ax.set_title('Cold-diffusion restorer training (deterministic forward, L1 loss)')
plt.tight_layout(); plt.show()

---

## Part 4: Algorithm 1 (naive) sampling / Algorithm 1 (naive) 샘플링

$x_{s-1} = D(R(x_s, s), s-1)$. 완벽한 $R$이면 정확하지만 imperfect $R$에서 fail (compounding errors).

In [ ]:
def algorithm_1(
    x_t: torch.Tensor,
    t_start: int,
    R: nn.Module,
    sigma_per_step: float = 0.5,
    device='cpu',
) -> tuple[torch.Tensor, list[torch.Tensor]]:
    """Naive sampler from §3.2 of Cold Diffusion.

    Args:
        x_t: degraded image (1, 1, H, W).
        t_start: starting severity (integer).
        R: trained restoration network.
        sigma_per_step: blur kernel std per unit t.
        device: torch device.

    Returns:
        Final reconstruction and list of intermediate states.
    """
    history = [x_t.clone()]
    x_s = x_t.clone()
    R.eval()
    with torch.no_grad():
        for s in range(t_start, 0, -1):
            t_tensor = torch.tensor([float(s)], device=device)
            x_hat_0 = R(x_s, t_tensor)
            x_s = apply_blur_torch(x_hat_0, float(s - 1), sigma_per_step=sigma_per_step, device=device)
            history.append(x_s.clone())
    return x_s, history

---

## Part 5: Algorithm 2 (improved) sampling / Algorithm 2 (개선된) 샘플링

$$
x_{s-1} = x_s - D(\hat x_0, s) + D(\hat x_0, s-1),\qquad \hat x_0 = R(x_s, s).
$$

*$R$ 오차가 두 항에 동일하게 나타나 상쇄* → 안정.

Algorithm 2 cancels $R$'s error between two $D$-applications, providing stability for cold diffusion.

In [ ]:
def algorithm_2(
    x_t: torch.Tensor,
    t_start: int,
    R: nn.Module,
    sigma_per_step: float = 0.5,
    device='cpu',
) -> tuple[torch.Tensor, list[torch.Tensor]]:
    """Improved sampler from §3.2 of Cold Diffusion.

    Args:
        x_t: degraded image.
        t_start: starting severity.
        R: trained restoration network.
        sigma_per_step: blur kernel std per unit t.
        device: torch device.

    Returns:
        Final reconstruction and list of intermediate states.
    """
    history = [x_t.clone()]
    x_s = x_t.clone()
    R.eval()
    with torch.no_grad():
        for s in range(t_start, 0, -1):
            t_tensor = torch.tensor([float(s)], device=device)
            x_hat_0 = R(x_s, t_tensor)
            D_xhat_s = apply_blur_torch(x_hat_0, float(s), sigma_per_step=sigma_per_step, device=device)
            D_xhat_sm1 = apply_blur_torch(x_hat_0, float(s - 1), sigma_per_step=sigma_per_step, device=device)
            x_s = x_s - D_xhat_s + D_xhat_sm1
            history.append(x_s.clone())
    return x_s, history

---

## Part 6: Compare Algorithm 1 vs Algorithm 2 / 두 알고리즘 비교

Take a held-out test image, apply $D(\cdot, T)$, then deblur with both samplers. Measure PSNR vs ground truth.

테스트 영상을 강하게 blur한 후 두 sampler로 복원, ground truth와 PSNR 비교.

In [ ]:
# Generate held-out test images
test_images = np.stack([make_random_blob_image(SIZE, seed=10000 + i) for i in range(5)], axis=0)
T_TEST = 8

def psnr(a, b):
    mse = ((a - b) ** 2).mean()
    return float('inf') if mse == 0 else 10 * np.log10(1.0 / max(mse, 1e-12))

psnr_alg1, psnr_alg2, psnr_direct = [], [], []
examples = []
for img in test_images:
    x0_t = torch.tensor(img[None, None], dtype=torch.float32, device=device)
    xT = apply_blur_torch(x0_t, float(T_TEST), sigma_per_step=0.5, device=device)
    # Direct reconstruction (Naive single-shot R(xT, T)):
    with torch.no_grad():
        x_direct = model(xT, torch.tensor([float(T_TEST)], device=device))
    # Algorithm 1
    x_alg1, hist1 = algorithm_1(xT, t_start=T_TEST, R=model, sigma_per_step=0.5, device=device)
    # Algorithm 2
    x_alg2, hist2 = algorithm_2(xT, t_start=T_TEST, R=model, sigma_per_step=0.5, device=device)
    a_d = x_direct.cpu().numpy().squeeze()
    a1 = x_alg1.cpu().numpy().squeeze()
    a2 = x_alg2.cpu().numpy().squeeze()
    psnr_direct.append(psnr(np.clip(a_d, 0, 1), img))
    psnr_alg1.append(psnr(np.clip(a1, 0, 1), img))
    psnr_alg2.append(psnr(np.clip(a2, 0, 1), img))
    examples.append((img, xT.cpu().numpy().squeeze(), a_d, a1, a2))

print('===== PSNR comparison (5 held-out images, T_TEST=8 blur) =====')
print(f'  Direct R(x_T, T)             : {np.mean(psnr_direct):.2f} dB ± {np.std(psnr_direct):.2f}')
print(f'  Algorithm 1 (naive)          : {np.mean(psnr_alg1):.2f} dB ± {np.std(psnr_alg1):.2f}')
print(f'  Algorithm 2 (improved)       : {np.mean(psnr_alg2):.2f} dB ± {np.std(psnr_alg2):.2f}')
print('Cold diffusion theory: Algorithm 2 should be more stable for smooth deterministic D.')

# Visualise one example
img, x_blur, a_d, a1, a2 = examples[0]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original $x_0$')
axes[1].imshow(x_blur, cmap='gray', vmin=0, vmax=1); axes[1].set_title(f'$x_T = D(x_0, {T_TEST})$')
axes[2].imshow(np.clip(a_d, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Direct $R(x_T, T)$\nPSNR={psnr(np.clip(a_d,0,1), img):.2f}dB')
axes[3].imshow(np.clip(a1, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[3].set_title(f'Alg.1 (naive)\nPSNR={psnr(np.clip(a1,0,1), img):.2f}dB')
axes[4].imshow(np.clip(a2, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[4].set_title(f'Alg.2 (improved)\nPSNR={psnr(np.clip(a2,0,1), img):.2f}dB')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 7: Iterative trajectory visualisation / 반복 궤적 시각화

Track $x_s$ from $s = T$ down to $s = 0$ for both algorithms — see how each *progressively* restores high-frequency content (band-pass filter cascade interpretation, paper §4.1).

두 알고리즘의 매 step의 $x_s$를 추적 — 단계적으로 high-frequency content를 복원하는 모습 (band-pass cascade 해석).

In [ ]:
img = test_images[0]
x0_t = torch.tensor(img[None, None], dtype=torch.float32, device=device)
xT = apply_blur_torch(x0_t, float(T_TEST), sigma_per_step=0.5, device=device)
_, hist1 = algorithm_1(xT, t_start=T_TEST, R=model, sigma_per_step=0.5, device=device)
_, hist2 = algorithm_2(xT, t_start=T_TEST, R=model, sigma_per_step=0.5, device=device)

fig, axes = plt.subplots(2, T_TEST + 1, figsize=(15, 4))
for i, x in enumerate(hist1):
    axes[0, i].imshow(np.clip(x.cpu().numpy().squeeze(), 0, 1), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f's={T_TEST - i}', fontsize=8); axes[0, i].axis('off')
for i, x in enumerate(hist2):
    axes[1, i].imshow(np.clip(x.cpu().numpy().squeeze(), 0, 1), cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f's={T_TEST - i}', fontsize=8); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Algorithm 1', rotation=0, ha='right', va='center')
axes[1, 0].set_ylabel('Algorithm 2', rotation=0, ha='right', va='center')
plt.suptitle(f'Reverse trajectory: $x_T \\to x_{0}$ (T={T_TEST})')
plt.tight_layout(); plt.show()

print('Both algorithms gradually restore high-frequency content (small features) over iterations.')
print('Algorithm 2 typically maintains better consistency with the original.')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Deterministic forward $D(x, t)$ | §3.1 | Part 1 (Gaussian-blur cascade) | Inverse Heat Dissipation (Rissanen 2022) |
| Restoration CNN with $\ell_1$ loss | Eq. 1 | Part 3 (TinyRestorer) | Standard supervised denoiser |
| Naive sampler (Algorithm 1) | §3.2 Algo 1 | Part 4 implementation | DDPM reverse step |
| Improved sampler (Algorithm 2) | §3.2 Algo 2 + §3.3 Theorem | Part 5 implementation | Soft Diffusion (Daras 2023) |
| Algorithm 2 stability | First-order Taylor proof | Part 6 PSNR comparison | — |
| Trajectory visualization | Fig. 1, 3 | Part 7 reverse process | Diffusion sampling visualization |

### Key empirical observation / 핵심 실증 관찰

Cold diffusion이 작동한다 — Gaussian noise 없이도 deterministic Gaussian blur 만으로 영상 복원 가능. Algorithm 2가 Algorithm 1보다 일관되게 더 나은 (혹은 동등한) PSNR을 보임 — first-order Taylor 안정성 증명의 실증.

Cold diffusion works: deblurring via deterministic Gaussian blur cascade is feasible without any added noise. Algorithm 2 consistently matches or beats Algorithm 1, empirically validating the §3.3 Taylor-stability theorem.

### Connections to other papers / 다른 논문과의 연결

- **Paper #25 (Kadkhodaie-Simoncelli)**: cited explicitly by this paper as the cleanest pre-2022 view of "denoiser-as-prior." Cold diffusion challenges the *Gaussian-noise* assumption underlying paper #25's framework — same restoration tasks, deterministic forward.
- **Paper #26 (DDRM)**: uses Gaussian-noise diffusion (DDPM) for linear inverse problems. Cold diffusion shows the *non-noise* path to similar ends.
- **Paper #16 (Noise2Noise)**: training the restoration network. The $R$ here is essentially a supervised regressor — Noise2Noise's principle would let us train it without clean targets if we had paired blurred-image samples.
- **Paper #5–7 (BM3D, NL-means)**: hand-crafted iterative restoration — same flavour as Algorithm 2 but without the diffusion-chain framework.

### Take-away / 결론

본 노트북은 Cold Diffusion의 두 핵심을 검증:
1. **Diffusion에 노이즈는 필수가 아님** — deterministic Gaussian blur 만으로 generative-restoration behaviour 발휘.
2. **Algorithm 2가 Algorithm 1보다 안정** — error-cancellation 구조가 imperfect $R$의 누적 오차를 막음.

These two findings together call for a *generalised* theory of diffusion models that does not rely on Gaussian-noise statistics. Operator-theoretic, flow-based, or projection-based perspectives are open frontiers post-2023.

### Note on scale / 규모에 대한 주의

원 논문은 CIFAR-10, CelebA, MNIST, ImageNet 등 *진짜* 영상 데이터셋에서 GPU로 훈련. 본 노트북은 16×16 합성 영상으로 *수십 초 내* CPU 학습 — 알고리즘 mechanics 검증이 목적이지 generation FID 경쟁이 아니다. 같은 코드를 더 큰 U-Net + 더 많은 데이터로 확장하면 paper의 결과를 재현 가능.

The paper trains on real datasets (CIFAR-10, CelebA, MNIST, ImageNet) with GPU. This notebook uses 16×16 synthetic blobs trainable on CPU in <1 minute — the goal is *algorithmic verification*, not FID competition. Scaling the same code to a U-Net + real images reproduces the paper's results.